# HORM Colab GPU runner (cymek-beta)

**Runtime: GPU (any Colab GPU runtime).** Runtime -> Change runtime type
-> **GPU** -> Run all. Expects ~30-60 min, ends with an auto-download.

What runs, in order (all commands are committed repo code, unmodified):
1. Fast gate: hormonal unit/integration/session tests (seconds).
2. HORM-003 prospective A/B (`--horm003 --force`: prior committed result
   is archived to `.previous`, never silently overwritten).
3. HORM-004 live-appraisal A/B (`--horm004 --force`, same archive rule).
4. Full test suite + import boundaries.
5. Packaging: every RESULT/manifest hashed, bundle zipped for download.

Head commit is recorded into the bundle at runtime for provenance.


In [ ]:
# CELL 0: FREEZE repo + environment (fail closed)
import hashlib
import os
import subprocess
import sys

REPO = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
BRANCH = "cymek-beta"
REPO_DIR = "/content/repo"

import os.path
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO, REPO_DIR],
                   check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "status", "--short"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tokenizers", "pytest"], check=True)

import torch
assert torch.cuda.is_available(), "HORM Colab run requires Google Colab GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

HEAD_SHA = subprocess.run(["git", "rev-parse", "HEAD"], check=True,
                          capture_output=True, text=True).stdout.strip()
print("HEAD_SHA:", HEAD_SHA)

def _sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

SPEC_SHA = _sha256_file("v5_contracts/model_spec.py")
GATE_SHA = _sha256_file("artifacts/v5/launch_readiness.json")
assert SPEC_SHA.upper().startswith("DFDCD883"), \
    "frozen model_spec.py drifted: " + SPEC_SHA
assert GATE_SHA.upper().startswith("95B2331A"), \
    "launch_readiness.json drifted: " + GATE_SHA
print("frozen model_spec.py + launch_readiness.json: VERIFIED")
print("HORM COLAB PREEXECUTION GATE: PASS")


In [ ]:
# CELL 1: RUN (committed commands only; --force archives prior results)
import os
import subprocess
import sys

ENV = dict(os.environ, PYTHONPATH="/content/repo")

def run(*args):
    print("\n=== ", " ".join(args), " ===", flush=True)
    subprocess.run([sys.executable, "-u", *args], check=True, env=ENV)

run("-m", "pytest", "tests/test_hormonal_state.py",
    "tests/test_hormonal_integration.py",
    "tests/test_hormonal_session.py", "-q")
run("experiments/HORM-001/run_horm002_ab.py", "--horm003",
    "--output", "experiments/HORM-001", "--force")
run("experiments/HORM-001/run_horm002_ab.py", "--horm004",
    "--output", "experiments/HORM-001", "--force")
run("-m", "pytest", "tests", "-q")
run("-m", "v5_contracts.import_boundaries")
print("\nALL HORM COLAB STAGES COMPLETE")


In [ ]:
# CELL 2: PACKAGE + DOWNLOAD (hash-bound bundle)
import glob
import hashlib
import json
import os
import shutil
import subprocess

HEAD_SHA = subprocess.run(["git", "rev-parse", "HEAD"], check=True,
                          capture_output=True, text=True).stdout.strip()
manifest = {"head_sha": HEAD_SHA, "files": { }}
for path in sorted(glob.glob("experiments/HORM-001/RESULT*.json")):
    digest = hashlib.sha256(open(path, "rb").read()).hexdigest()
    document = json.load(open(path, encoding="utf-8"))
    manifest["files"][os.path.basename(path)] = {
        "sha256": digest,
        "verdict": document.get("verdict"),
        "result_sha256": document.get("sha256"),
    }
    print(os.path.basename(path), "->", document.get("verdict"))
with open("experiments/HORM-001/COLAB_BUNDLE_MANIFEST.json", "w",
           encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2, sort_keys=True)
    handle.write("\n")
shutil.make_archive("/content/HORM-COLAB-RESULTS", "zip",
                    "experiments/HORM-001")
print("bundle:", "/content/HORM-COLAB-RESULTS.zip")
try:
    from google.colab import files
    files.download("/content/HORM-COLAB-RESULTS.zip")
except Exception as exc:
    print("manual download from experiments/HORM-001/:", exc)
